## Docling + LangChain: Loading and Chunking Documents

#### 1. What is Docling and how to use `langchain-docling`

[Docling](https://github.com/docling-project/docling) parses PDF, DOCX, PPTX, HTML, and other formats into a rich unified representation (document layout, tables, etc.), making them ready for generative AI workflows like RAG.

The `langchain-docling` package exposes Docling through the `DoclingLoader` document loader (`langchain_docling.loader`).

| Property | Value |
|---|---|
| Package | `langchain-docling` |
| Local | ✅ |
| Serializable | ❌ |
| JS support | ❌ |

#### 2. Chunking with Docling

Chunking should respect the **token limit of the embedding model**. Docling provides two chunkers:

- **`HierarchicalChunker`**: creates chunks based primarily on document structure and elements, retaining associated heading information. Useful for fine-grained, structure-preserving chunks. *Best for: element-level extraction and detailed metadata.*
- **`HybridChunker`**: builds on hierarchical chunking, then splits oversized chunks according to a token budget and merges suitable undersized chunks. Generally a good starting point for RAG. *Best for: retrieval with embedding models that have token limits.*

Key points:
- `HybridChunker` supports a maximum token budget.
- `HierarchicalChunker` does not have a fixed chunk-size setting.

| Property | `HierarchicalChunker` | `HybridChunker` |
|---|---|---|
| How chunks are formed | Document elements | Document elements plus token-aware refinement |
| Fixed maximum token size | No | Yes, configurable |
| Splits oversized elements | No token-budget splitting | Yes |
| Merges neighboring chunks | Normally only list-item merging | Yes, when compatible and enabled |
| Best use | Fine-grained structural extraction | General RAG ingestion |

#### 3. Decision: use `HybridChunker`

A document can put a huge amount of text into a single chunk if we use `HierarchicalChunker`, and our small embedding model and LLM may fail to handle chunks that large. `HybridChunker` keeps every chunk within a token budget.

#### 4. Does Docling have a default tokenizer?

Yes. If you don't specify a tokenizer:

```python
chunker = HybridChunker()
```

Docling uses its default Hugging Face tokenizer, which is based on:

```
sentence-transformers/all-MiniLM-L6-v2
```

In the example below the tokenizer is passed explicitly, set to the same model name as the embedding model, so token counts match the model that will embed the chunks.

In [1]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType, HybridChunker
from IPython.display import display, Markdown

### Configure the chunker and load the page

- `tokenizer`: the embedding model whose tokenizer defines the token budget
- `merge_peers=True`: merge undersized neighboring chunks that share the same headings
- `repeat_table_header=True`: repeat the table header when a table is split across chunks
- `export_type=ExportType.DOC_CHUNKS`: return one LangChain `Document` per chunk

In [2]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
chunker = HybridChunker(
    tokenizer=EMBED_MODEL,
    merge_peers=True,
    repeat_table_header=True
)

loader = DoclingLoader(
    file_path="https://docs.langchain.com/oss/python/integrations/document_loaders/docling",
    export_type=ExportType.DOC_CHUNKS,
    chunker=chunker
)
docs = loader.load()

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Token indices sequence length is longer than the specified maximum sequence length for this model (1349 > 512). Running this sequence through the model will result in indexing errors


> **Note:** the warning `Token indices sequence length is longer than the specified maximum sequence length for this model (1349 > 512)...` can be ignored. It appears during chunking because the tokenizer's model cannot handle more than its token limit at a time, but `HybridChunker` then splits the text, so the final chunks are always within the limit.

#### 6. Inspecting the output

Each returned `Document` has `page_content` plus `metadata` containing the source and Docling's `dl_meta` (schema, `doc_items`, `headings`, `origin`).

In [3]:
print("Length of Documents retrieved: ", len(docs))
print("Page content:\n", docs[0])
print("\nMetadata:\n", docs[0].metadata)
print("\nKeys on metadata:\n", docs[0].metadata.keys())
print("\ndl_meta keys:\n", docs[0].metadata['dl_meta'].keys())

Length of Documents retrieved:  31
Page content:
 page_content='Popular Providers
- [OpenAI](/oss/python/integrations/providers/openai)
- [Anthropic](/oss/python/integrations/providers/anthropic)
- [Google](/oss/python/integrations/providers/google)
- [AWS](/oss/python/integrations/providers/aws)
- [NVIDIA](/oss/python/integrations/providers/nvidia)
- [Hugging Face](/oss/python/integrations/providers/huggingface)
- [Microsoft](/oss/python/integrations/providers/microsoft)
- [Ollama](/oss/python/integrations/providers/ollama)
- [Groq](/oss/python/integrations/providers/groq)
- [Fireworks](/oss/python/integrations/providers/fireworks)
- [Browserbase integrations](/oss/python/integrations/providers/browserbase)' metadata={'source': 'https://docs.langchain.com/oss/python/integrations/document_loaders/docling', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/28', 'parent': {'$ref': '#/groups/6'}, 'children': [], '

### Rendering the first few chunks as Markdown

In [4]:
for doc in docs[:5]:
    display(Markdown(doc.page_content))

Popular Providers
- [OpenAI](/oss/python/integrations/providers/openai)
- [Anthropic](/oss/python/integrations/providers/anthropic)
- [Google](/oss/python/integrations/providers/google)
- [AWS](/oss/python/integrations/providers/aws)
- [NVIDIA](/oss/python/integrations/providers/nvidia)
- [Hugging Face](/oss/python/integrations/providers/huggingface)
- [Microsoft](/oss/python/integrations/providers/microsoft)
- [Ollama](/oss/python/integrations/providers/ollama)
- [Groq](/oss/python/integrations/providers/groq)
- [Fireworks](/oss/python/integrations/providers/fireworks)
- [Browserbase integrations](/oss/python/integrations/providers/browserbase)

Integrations by component
- [Chat models](/oss/python/integrations/chat)
- [Tools and toolkits](/oss/python/integrations/tools)
- [Middleware](/oss/python/integrations/middleware)
- [Sandboxes](/oss/python/integrations/sandboxes)
- [Backends](/oss/python/integrations/backends)
- [Checkpointers](/oss/python/integrations/checkpointers)
- [Stores](/oss/python/integrations/long-term-memory)
- [Retrievers](/oss/python/integrations/retrievers)
- [Text splitters](/oss/python/integrations/splitters)
- [Embedding models](/oss/python/integrations/embeddings)
- [Vector stores](/oss/python/integrations/vectorstores)
- [Document loaders](/oss/python/integrations/document_loaders)

On this page
- [Overview](#overview)
- [Integration details](#integration-details)
- [Loader features](#loader-features)
- [Setup](#setup)
- [Initialization](#initialization)
- [Load](#load)
- [Lazy load](#lazy-load)
- [End-to-end example](#end-to-end-example)
- [Ingestion](#ingestion)
- [RAG](#rag)
- [API reference](#api-reference)

Docling integration
Copy page
Integrate with the Docling document loader using LangChain Python.
Copy page
[Docling](https://github.com/docling-project/docling)
parses PDF, DOCX, PPTX, HTML, and other formats into a rich unified representation including document layout, tables etc., making them ready for generative AI workflows like RAG. This integration provides Docling's capabilities via the
```
DoclingLoader
```
document loader.

Docling integration
 Overview
 Integration details
langchain_docling.loader, Package = langchain-docling. langchain_docling.loader, Local = ✅. langchain_docling.loader, Serializable = ❌. langchain_docling.loader, JS support = ❌

## Summary

- Docling turns PDF, DOCX, PPTX, HTML, etc. into a structured representation; `DoclingLoader` plugs it into LangChain.
- `HierarchicalChunker` preserves structure but has no chunk-size limit; `HybridChunker` adds a token budget (split oversized, merge undersized).
- DeepDive uses **`HybridChunker`** so chunks fit the small embedding model and LLM.
- If no tokenizer is given, Docling defaults to the `sentence-transformers/all-MiniLM-L6-v2` tokenizer.